# Ejercicio 4 — Aprendizaje Profundo
## Clasificación de residuos para reciclaje (CNN desde cero vs Transfer Learning)

Este notebook es la **capa narrativa** del entregable: cada celda es una llamada a
`pipeline/` más su explicación, sin lógica de negocio (la lógica vive en los
módulos). Sirve además como entorno de ejecución en **Google Colab con GPU**.

**Problema y motivación.** Se clasifican imágenes de residuos en 5 categorías
(`cardboard`, `glass`, `metal`, `paper`, `plastic`) para apoyar la separación
automática en reciclaje. Es un problema de **clasificación** (no detección): cada
imagen pertenece a una sola clase.

**Dataset.** TrashNet (garythung/trashnet, Stanford), versión redimensionada
512×384, descargada desde GitHub sin autenticación. Se usan 5 de las 6 clases
(se descarta `trash`), cada una recortada a 400 imágenes → 2000 balanceadas.

## 0. Entorno (Colab con GPU / local)

In [ ]:
# Funciona en Google Colab (clona el repo + instala deps) y en local (ya en el repo).
import os, sys, subprocess

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Gonzalo-Romero-V/EJML---04-Aprendizaje-Profundo.git"
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if EN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt"], check=True)

import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

> En Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**
> antes de ejecutar. Si la lista de GPU sale vacía, el entrenamiento correrá en CPU
> (mucho más lento).

## Parte A — Dataset

Descarga el zip de TrashNet, selecciona las 5 clases, recorta a 400 por clase y
arma `data/dataset/<clase>/`. El código de descarga es parte del entregable.

In [ ]:
!python scripts/preparar_dataset.py

In [ ]:
# Una imagen de muestra por clase (verificación visual del dataset).
import matplotlib.pyplot as plt
from PIL import Image
from pipeline import config

fig, axes = plt.subplots(1, len(config.CLASES), figsize=(15, 3))
for ax, clase in zip(axes, config.CLASES):
    muestra = next((config.DATASET_DIR / clase).iterdir())
    ax.imshow(Image.open(muestra)); ax.set_title(clase); ax.axis("off")
plt.show()

## Parte B — CNN desde cero

Se carga el dataset con split 80/20 y **data augmentation solo en train** (flip,
rotación, zoom). La normalización (Rescaling 1/255) va dentro del modelo.

In [ ]:
import time
import numpy as np
import pandas as pd
from IPython.display import Image as Img
from pipeline import data, cnn, transfer, evaluation as ev, audit, plotting

tr, va, clases = data.cargar_datasets()
test_ds, _ = data.cargar_test_crudo()
print("clases:", clases)

**Arquitectura.** 3 bloques convolucionales (Conv-BN-Conv-BN-MaxPool-Dropout) con
32, 64 y 128 filtros (se duplican al reducir resolución para representar patrones
cada vez más compuestos), kernel 3×3, BatchNormalization (estabiliza el
entrenamiento) y Dropout 0.25 (regulariza). La cabeza usa Global Average Pooling
(menos parámetros y menos sobreajuste que Flatten), una densa de 128, Dropout 0.5
y softmax de 5 clases.

In [ ]:
modelo_cnn = cnn.construir_cnn()
modelo_cnn.summary()

In [ ]:
t0 = time.time()
hist_cnn = modelo_cnn.fit(tr, validation_data=va, epochs=config.CNN_EPOCHS,
                          callbacks=cnn.callbacks_entrenamiento(), verbose=2)
t_cnn = time.time() - t0
Img(str(plotting.curvas_entrenamiento(hist_cnn.history, "cnn_curvas.png",
                                      "CNN propia — entrenamiento")))

In [ ]:
# Métricas por clase y matriz de confusión sobre el test.
yt, yp, _ = ev.predecir(modelo_cnn, test_ds)
rep_cnn = ev.reporte_clasificacion(yt, yp, clases)
display(pd.DataFrame(rep_cnn).T.round(3))
cm_cnn = ev.matriz_confusion(yt, yp, len(clases))
Img(str(plotting.matriz_confusion_fig(cm_cnn, clases, "confusion_cnn.png",
                                      "Matriz de confusión — CNN propia")))

**Grad-CAM.** Resalta las zonas de la imagen que la red usa para decidir. Si el
mapa se enciende sobre el objeto (no sobre el fondo), la red aprendió patrones
reales. Se muestran 3 imágenes de test bien clasificadas.

In [ ]:
imgs, ytg, ypg = audit.recolectar_predicciones(modelo_cnn, test_ds)
sel = np.flatnonzero(ytg == ypg)[:3]
Img(str(plotting.grid_gradcam(modelo_cnn, imgs[sel], clases, ytg[sel], ypg[sel],
                              "gradcam_cnn.png")))

## Parte C — Transfer Learning con MobileNetV2

**Fase 1 (Feature Extraction):** base congelada, se entrena solo la cabeza nueva
(lr alto 1e-3). **Fase 2 (Fine-Tuning):** se descongelan las últimas 30 capas con
lr muy bajo (1e-5) para ajustar sin destruir los pesos preentrenados.

In [ ]:
modelo_tl, base = transfer.construir_transfer()
transfer.compilar_fase1(modelo_tl)
print("Params entrenables Fase 1:", ev.params_entrenables(modelo_tl))
t0 = time.time()
h1 = modelo_tl.fit(tr, validation_data=va, epochs=config.TL_FE_EPOCHS,
                   callbacks=transfer.callbacks_entrenamiento(), verbose=2)
t_f1 = time.time() - t0
yt1, yp1, _ = ev.predecir(modelo_tl, test_ds)

In [ ]:
transfer.preparar_fase2(modelo_tl, base)
print("Params entrenables Fase 2:", ev.params_entrenables(modelo_tl))
t0 = time.time()
h2 = modelo_tl.fit(tr, validation_data=va, epochs=config.TL_FT_EPOCHS,
                   callbacks=transfer.callbacks_entrenamiento(), verbose=2)
t_f2 = time.time() - t0
yt2, yp2, _ = ev.predecir(modelo_tl, test_ds)
hist_tl = {k: h1.history[k] + h2.history.get(k, []) for k in h1.history}
Img(str(plotting.curvas_entrenamiento(hist_tl, "tl_curvas.png",
        "MobileNetV2 — Fase 1 + Fase 2",
        linea_finetuning=len(h1.history["accuracy"]))))

In [ ]:
cm_tl = ev.matriz_confusion(yt2, yp2, len(clases))
Img(str(plotting.matriz_confusion_fig(cm_tl, clases, "confusion_tl.png",
                                      "Matriz de confusión — MobileNetV2 Fase 2")))

**Tabla comparativa de los 3 modelos.** Nota la diferencia de parámetros
entrenables: MobileNetV2 Fase 1 logra buen accuracy ajustando muy pocos
parámetros, porque reutiliza las características aprendidas en ImageNet.

In [ ]:
filas = [
    ev.fila_comparativa("CNN propia", yt, yp, tiempo_s=t_cnn, modelo=modelo_cnn),
    ev.fila_comparativa("MobileNetV2 Fase 1", yt1, yp1, tiempo_s=t_f1, modelo=modelo_tl),
    ev.fila_comparativa("MobileNetV2 Fase 2", yt2, yp2, tiempo_s=t_f2, modelo=modelo_tl),
]
df = ev.tabla_comparativa(filas)
display(df)
Img(str(plotting.tabla_comparativa_fig(df, "tabla_comparativa.png")))

## Parte D — Auditoría y mejora

Análisis crítico de los propios modelos: par de clases más confundido, intento de
mejora del modelo más débil, e imágenes mal clasificadas por el mejor modelo.

In [ ]:
ca, cb, n_conf = audit.par_mas_confundido(cm_cnn, clases)
print(f"Par más confundido por la CNN: {ca} <-> {cb} ({n_conf} casos)")

**Dos mejoras** sobre la CNN propia (el modelo de menor F1): regularización **L2**
(mejora 1) y un **schedule de learning rate** con ReduceLROnPlateau (mejora 2). Se
documenta honestamente si mejoraron o no.

In [ ]:
f1_por_modelo = {f["modelo"]: f["f1_macro"] for f in filas}
peor = min(f1_por_modelo, key=f1_por_modelo.get)
print("Modelo con menor F1:", peor, round(f1_por_modelo[peor], 3))

modelo_mej = audit.construir_cnn_mejorada()
modelo_mej.fit(tr, validation_data=va, epochs=config.CNN_EPOCHS,
               callbacks=audit.callbacks_mejorados(), verbose=2)
ytm, ypm, _ = ev.predecir(modelo_mej, test_ds)
f1_mej = ev.reporte_clasificacion(ytm, ypm, clases)["macro avg"]["f1-score"]
f1_base = rep_cnn["macro avg"]["f1-score"]
print(f"F1 CNN base {f1_base:.3f} -> mejorada {f1_mej:.3f} "
      f"({'mejoró' if f1_mej > f1_base else 'no mejoró'})")

In [ ]:
# Imágenes mal clasificadas por el mejor modelo global.
mejor = max(f1_por_modelo, key=f1_por_modelo.get)
usar = (modelo_cnn, yt, yp) if mejor == "CNN propia" else (modelo_tl, yt2, yp2)
imb, ytb, ypb = audit.recolectar_predicciones(usar[0], test_ds)
err = audit.indices_mal_clasificados(ytb, ypb, maximo=8)
titulos = [f"{clases[ytb[i]]}->{clases[ypb[i]]}" for i in err]
print(f"Mejor modelo: {mejor} | total mal clasificadas: {int((ytb != ypb).sum())}")
Img(str(plotting.grid_imagenes(imb[err], titulos, "errores_mejor_modelo.png",
                               f"Mal clasificadas — {mejor}")))

In [ ]:
# Clase con menor Recall (pregunta técnica 18).
clase_r, recall_r = ev.clase_menor_recall(rep_cnn, clases)
print(f"Clase con menor Recall en la CNN: {clase_r} ({recall_r:.3f})")

## Conclusión

A completar tras la corrida en GPU con las métricas reales: qué modelo se
recomendaría en producción y bajo qué condiciones (precisión requerida, costo de
cómputo, latencia), conectando con la diferencia de parámetros entrenables y la
ventaja del Transfer Learning frente a entrenar una CNN desde cero con un dataset
pequeño.